# 5.22 Nonparametric Bayes & Dirichlet processes

**Lesson tagline.** A Dirichlet process lets the data decide how many clusters are needed.

Dirichlet processes extend Bayesian inference beyond a fixed number of components. The Chinese restaurant predictive rule gives old clusters mass by count and reserves mass for genuinely new clusters. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 522
rng = np.random.default_rng(SEED)

def normalize(values):
    arr = np.asarray(values, dtype=float)
    total = arr.sum()
    if total <= 0:
        raise ValueError("positive mass required")
    return arr / total

def crp_predictive(counts, alpha):
    counts = np.asarray(counts, dtype=float)
    total = counts.sum() + alpha
    return np.append(counts / total, alpha / total)

def expected_clusters(alpha, n):
    return sum(alpha / (alpha + i) for i in range(n))

def run_crp(alpha, n, random_state):
    counts = []
    assignments = []
    for _ in range(n):
        probs = crp_predictive(counts, alpha) if counts else np.array([1.0])
        choice = random_state.choice(len(probs), p=probs)
        if choice == len(counts):
            counts.append(1)
        else:
            counts[choice] += 1
        assignments.append(choice)
    return np.array(assignments), np.array(counts)

def build_dp_ladder():
    ladder = []
    ladder.append({"name": "D1 two occupied clusters", "counts": [3, 1], "alpha": 2.0, "base_mean": 0.5})
    ladder.append({"name": "D2 four-state seating", "counts": [5, 2, 1], "alpha": 1.5, "base_mean": 0.45})
    ladder.append({"name": "D3 bimodal ambiguity", "counts": [12, 10], "alpha": 4.0, "base_mean": 0.5})
    ladder.append({"name": "D4 real-ish 2-D table", "counts": [18, 8, 6, 2], "alpha": 2.5, "base_mean": 0.35})
    ladder.append({"name": "D5 high-dim concentration stress", "counts": [35, 12, 8, 4, 2], "alpha": 8.0, "base_mean": 0.25})
    return ladder

def predictive_tv_curve(counts, alpha, reference_alpha, max_customers):
    counts = list(counts)
    ref_counts = list(counts)
    values = []
    for _ in range(max_customers):
        p = crp_predictive(counts, alpha)
        q = crp_predictive(ref_counts, reference_alpha)
        m = max(len(p), len(q))
        p = np.pad(p, (0, m - len(p)))
        q = np.pad(q, (0, m - len(q)))
        values.append(0.5 * np.sum(np.abs(p - q)))
        counts[0] += 1
        ref_counts[0] += 1
    return values

## The concept, built once (D1)
The Chinese restaurant predictive rule is
$$P(z_{n+1}=k\mid z_{1:n})=\frac{n_k}{n+\alpha},\quad P(z_{n+1}=\text{new})=\frac{\alpha}{n+\alpha}.$$
Counts get old-cluster mass; the concentration parameter keeps a new-cluster option alive.

In [ ]:
counts = np.array([3, 1])
alpha = 2.0
probs = crp_predictive(counts, alpha)
new_after_n = alpha / (alpha + 100)
assert np.allclose(probs, [0.5, 1.0 / 6.0, 1.0 / 3.0])
assert abs(new_after_n - 2.0 / 102.0) < 1e-12
print("predictive", probs)
print("new after 100", new_after_n)

The base distribution shapes new clusters. The lesson's predictive mean mixes old cluster means and the base mean: $(3\cdot0.2+1\cdot0.8+2\cdot0.5)/6=0.400$. The cited expected number of clusters at $n=100$ lies between $8$ and $9$.

In [ ]:
cluster_means = np.array([0.2, 0.8])
base_mean = 0.5
predictive_mean = (3 * 0.2 + 1 * 0.8 + 2 * base_mean) / 6
expected_k_100 = expected_clusters(2.0, 100)
assert abs(predictive_mean - 0.4) < 1e-12
assert 8.0 < expected_k_100 < 9.0
print("predictive mean", predictive_mean)
print("expected clusters at n=100", expected_k_100)

## The dataset ladder
The F10 ladder moves from exact CRP counts to seating sequences, ambiguous mixtures, a small 2-D clustering table, and a high-dimensional concentration stress test.

In [ ]:
ladder = build_dp_ladder()
for i, rung in enumerate(ladder, start=1):
    preview_assignments, preview_counts = run_crp(rung["alpha"], 8, rng)
    print(i, rung["name"], "counts=", rung["counts"], "alpha=", rung["alpha"])
    print("simulated seating", preview_assignments, "occupied=", len(preview_counts))

## Run the SAME method across D1-D5
Collect the plan metric on every rung from D1–D5 with the same algorithmic implementation, then print a compact per-rung table.

In [ ]:
customer_grid = [1, 2, 5, 10, 20, 40]
dp_results = {}
print("rung | final predictive TV | new-cluster prob")
for rung in ladder:
    reference_alpha = 2.0
    curve = predictive_tv_curve(rung["counts"], rung["alpha"], reference_alpha, customer_grid[-1])
    probs = crp_predictive(rung["counts"], rung["alpha"])
    dp_results[rung["name"]] = {"curve": curve, "probs": probs}
    print(f"{rung['name']} | {curve[-1]:.4f} | {probs[-1]:.3f}")

## Results visualization
The closing figure has target/sample panels on top and the requested error-vs-iteration or error-vs-sample curve below.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, rung in zip(axes[0], ladder):
    probs = dp_results[rung["name"]]["probs"]
    labels = list(range(len(probs) - 1)) + ["new"]
    ax.bar(range(len(probs)), probs)
    ax.set_xticks(range(len(probs)))
    ax.set_xticklabels(labels, rotation=45)
    ax.set_title(rung["name"], fontsize=8)
for ax, rung in zip(axes[1], ladder):
    curve = dp_results[rung["name"]]["curve"]
    ax.plot(np.arange(1, len(curve) + 1), curve)
    ax.set_title("predictive TV", fontsize=8)
fig.tight_layout()
plt.show()

## Pitfall on the hardest rung
The D5 pitfall is overreading $lpha$ and ignoring the base distribution. A large $lpha$ creates many novelty proposals, but if the base measure is wrong those new clusters are poor. The fix is a sensitivity sweep over $lpha$ and base means.

In [ ]:
d5 = ladder[-1]
for trial_alpha, trial_base in [(20.0, 0.9), (8.0, 0.25), (2.0, 0.25)]:
    probs = crp_predictive(d5["counts"], trial_alpha)
    occupied_means = np.linspace(0.1, 0.7, len(d5["counts"]))
    numerator = np.dot(d5["counts"], occupied_means) + trial_alpha * trial_base
    predictive_mean = numerator / (np.sum(d5["counts"]) + trial_alpha)
    print("alpha", trial_alpha, "base", trial_base, "p_new", round(float(probs[-1]), 3), "mean", round(float(predictive_mean), 3))

## Evaluate it + Practice
- **Metric.** Predictive total variation and marginal cluster-assignment error.
- **No-skill baseline.** Always assign to the largest existing cluster.
- **Cheap sanity check.** Predictive probabilities must sum to 1 and finite data occupies finitely many clusters.
- **Ablation.** Change $lpha$ without changing the base measure and compare novelty quality.
- **Failure signals.** Too many singleton clusters, zero new-cluster mass, or novel clusters with implausible parameters.

### Practice prompts
1. Simulate CRP seating for D2 and compare occupied clusters to expectation.
2. Sweep $lpha$ for D5 and plot new-cluster probability.
3. Change the base mean and recompute posterior predictive means.

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here